%md
# Eval 03 · Agent scoring
Scores the agent's 20 triage decisions against the labels: decisions, patterns, ranking, cost, and a projection to the full alert queue.

In [0]:
from pyspark.sql import functions as F

EVAL = "workspace.aml_eval"

raw = spark.read.json("/Volumes/workspace/aml_eval/eval_files/agent_results.jsonl")

results = raw.select(
    "alert_id",
    "model",
    F.col("decision.decision").alias("agent_decision"),
    F.col("decision.pattern").alias("agent_pattern"),
    F.col("decision.confidence").alias("agent_confidence"),
    F.col("decision.risk_score").cast("int").alias("risk_score"),
    "tokens_in",
    "tokens_out",
    "cost_usd",
    "seconds",
)

scored = (
    spark.table(f"{EVAL}.agent_sample")
    .join(results, on="alert_id", how="left")
    .withColumn("is_laundering", F.col("sample_group") == "laundering")
    .withColumn("escalated", F.col("agent_decision") == "ESCALATE")
    .withColumn("pattern_correct", F.array_contains("true_patterns", F.col("agent_pattern")))
)

print(f"Rows: {scored.count()}, missing results: {scored.filter(F.col('agent_decision').isNull()).count()}")

(
    scored.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{EVAL}.agent_results")
)

display(scored.select("alert_id", "sample_group", "agent_decision", "agent_pattern", "true_patterns", "agent_confidence", "risk_score"))

In [0]:
p = scored.toPandas()

tp = int(((p.is_laundering) & (p.escalated)).sum())
fn = int(((p.is_laundering) & (~p.escalated)).sum())
fp = int(((~p.is_laundering) & (p.escalated)).sum())
tn = int(((~p.is_laundering) & (~p.escalated)).sum())

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn)
specificity = tn / (tn + fp)
accuracy = (tp + tn) / len(p)

print("Confusion matrix (rows = truth, columns = agent)")
print(f"                  ESCALATE  CLOSE")
print(f"  laundering        {tp:>4}    {fn:>4}")
print(f"  false positive    {fp:>4}    {tn:>4}")
print()
print(f"Accuracy:     {accuracy:.0%}")
print(f"Precision:    {precision:.0%}   (of escalations, share that are real)")
print(f"Recall:       {recall:.0%}   (of real cases, share escalated)")
print(f"Specificity:  {specificity:.0%}   (of false alerts, share correctly closed)")

In [0]:
laundering = p[p.is_laundering]
escalated_laundering = laundering[laundering.escalated]
single = laundering[laundering.true_patterns.apply(len) == 1]

print(f"Pattern accuracy, all laundering alerts:       {laundering.pattern_correct.mean():.0%}  (n={len(laundering)})")
print(f"Pattern accuracy, escalated laundering only:   {escalated_laundering.pattern_correct.mean():.0%}  (n={len(escalated_laundering)})")
print(f"Pattern accuracy, single-pattern accounts:     {single.pattern_correct.mean():.0%}  (n={len(single)})")

pos = p[p.is_laundering].risk_score.tolist()
neg = p[~p.is_laundering].risk_score.tolist()
pairs = [(a > b) + 0.5 * (a == b) for a in pos for b in neg]
auc = sum(pairs) / len(pairs)

print()
print(f"Mean risk score, laundering:      {sum(pos) / len(pos):.0f}")
print(f"Mean risk score, false positive:  {sum(neg) / len(neg):.0f}")
print(f"Ranking AUC (risk_score):         {auc:.2f}  (0.5 = random, 1.0 = perfect)")

In [0]:
ALERTS = 5473
RULE_TP = 465
RULE_FP = ALERTS - RULE_TP

projected_tp = recall * RULE_TP
projected_fp = (1 - specificity) * RULE_FP
projected_escalations = projected_tp + projected_fp
projected_precision = projected_tp / projected_escalations if projected_escalations else 0

print(f"Average cost per alert:   ${p.cost_usd.mean():.3f}")
print(f"Average time per alert:   {p.seconds.mean():.0f}s")
print(f"Cost for the full queue:  ${p.cost_usd.mean() * ALERTS:,.0f}")
print()
print("Projection to the full queue (rules alone: 8.5% precision)")
print(f"  Escalated by agent:     {projected_escalations:,.0f} of {ALERTS:,} ({projected_escalations / ALERTS:.0%})")
print(f"  Projected precision:    {projected_precision:.0%}")
print(f"  Real cases kept:        {projected_tp:,.0f} of {RULE_TP} ({recall:.0%})")

In [0]:
MINUTES_PER_ALERT = 30
MINUTES_WITH_AGENT_EVIDENCE = 10

hours_today = ALERTS * MINUTES_PER_ALERT / 60

hours_filter = projected_escalations * MINUTES_PER_ALERT / 60
missed_filter = RULE_TP - projected_tp

hours_two_tier = (
    projected_escalations * MINUTES_PER_ALERT
    + (ALERTS - projected_escalations) * MINUTES_WITH_AGENT_EVIDENCE
) / 60

agent_cost = p.cost_usd.mean() * ALERTS

print(f"Today (rules only):        {hours_today:,.0f} analyst hours, all {RULE_TP} real cases reviewed")
print()
print(f"A) Agent as auto-filter:   {hours_filter:,.0f} hours ({1 - hours_filter / hours_today:.0%} saved)")
print(f"                           but {missed_filter:,.0f} real cases never seen by a human. Not acceptable.")
print()
print(f"B) Two-tier (recommended): {hours_two_tier:,.0f} hours ({1 - hours_two_tier / hours_today:.0%} saved)")
print(f"                           every alert still reviewed by a human; escalations fast-tracked")
print()
print(f"Agent cost for the queue:  ${agent_cost:,.0f}")

metrics = [
    ("rules_precision", 0.085),
    ("agent_precision_sample", precision),
    ("agent_projected_precision", projected_precision),
    ("agent_recall", recall),
    ("agent_specificity", specificity),
    ("agent_accuracy", accuracy),
    ("pattern_accuracy_escalated", escalated_laundering.pattern_correct.mean()),
    ("ranking_auc", auc),
    ("cost_per_alert_usd", p.cost_usd.mean()),
    ("seconds_per_alert", p.seconds.mean()),
    ("alerts_total", ALERTS),
    ("projected_escalations", projected_escalations),
    ("hours_today", hours_today),
    ("hours_two_tier", hours_two_tier),
    ("hours_saved_two_tier", hours_today - hours_two_tier),
    ("agent_cost_full_queue_usd", agent_cost),
]

summary = spark.createDataFrame(
    [(name, float(value)) for name, value in metrics],
    "metric STRING, value DOUBLE",
)

(
    summary.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{EVAL}.eval_summary")
)

display(summary)